# Sentiment Analysis via Fine-Tuning DistilBERT
This notebook fine-tunes a pre-trained DistilBERT model for binary text classification (sentiment analysis) using the IMDb movie reviews dataset. It is designed to be run in Google Colab.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
%pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# 1. Load Dataset
print("Loading IMDB dataset...")
dataset = load_dataset("imdb")

# For faster training in a mini-project, we'll use a small subset
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
# 2. Tokenization
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

In [ ]:
# 3. Load Pre-trained Model
print("Loading pre-trained DistilBERT model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# 4. Define Evaluation Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels)["f1"]
    
    return {"accuracy": accuracy, "f1": f1}

In [ ]:
# 5. Training Arguments & Trainer
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

In [ ]:
# 6. Evaluate Model
print("Evaluating on test set...")
results = trainer.evaluate()
print(results)

In [ ]:
# 7. Save Model
save_path = "./sentiment_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

In [ ]:
# 8. Test on Custom Input
from transformers import pipeline

# Load the fine-tuned model into a pipeline
sentiment_pipeline = pipeline("text-classification", model=save_path, tokenizer=save_path, device=0 if device=="cuda" else -1)

# Custom test sentences
texts = [
    "This movie was an absolute waste of time. I hated it.",
    "A brilliant masterpiece with phenomenal acting!",
    "It was okay, but the ending could have been better."
]

print("\n--- Custom Predictions ---")
for text in texts:
    prediction = sentiment_pipeline(text)
    label = "Positive" if prediction[0]['label'] == 'LABEL_1' else "Negative"
    score = prediction[0]['score']
    print(f"Text: '{text}'")
    print(f"Prediction: {label} (Confidence: {score:.4f})\n")